In [2]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

### new segment

In [29]:
cx_data0 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        segment,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data0 = pd.read_sql(cx_data0, conn4)
cx_data0.head()

,segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,643553,532162,591490,702954,666446,728138,749421,745258,736039,729555,707487,714627,682575,8929705,22.74,97648,89625,95374,102849,102923,104776,105139,105044,104740,104081,103306,102573,100448,108852,2.53
1,2-WEEKLY,657600,565326,625494,699825,667488,707954,719626,732853,733061,711421,700828,709856,697665,8928997,22.74,224448,203329,219268,238912,235632,240278,241972,243866,243953,240134,238636,235743,232387,296287,6.88
2,3-MONTHLY,1213231,1070538,1149882,1244925,1243132,1301369,1350608,1487036,1432724,1328451,1316692,1333680,1379189,16851457,42.92,628436,565455,609552,654022,662834,679912,698727,761491,729077,675556,671598,663911,678514,2147189,49.85
3,4-OTHER,268015,234211,219685,180279,168880,191867,196781,195535,298530,367141,407326,468705,529159,3726114,9.49,156185,139532,135952,118278,112685,123395,126465,129718,185755,207502,227153,248263,274815,1612817,37.45
4,5-INACTIVE,60001,54443,58960,58965,57077,60997,63744,67498,67772,67238,67346,69411,72061,825513,2.10,29300,27499,29900,28043,27708,28598,30092,31685,32282,31443,31948,32647,34158,141914,3.29


In [4]:
cx_data0

,segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,352220,303092,334537,379730,360208,386012,396721,392675,388656,387573,377496,378049,364055,4801024,12.23,43049,40871,42892,44427,44276,44513,44625,44579,44522,44420,44274,43956,43294,45177,1.05
1,2-WEEKLY,939622,786050,873780,1012242,960050,1031207,1051170,1064398,1060765,1034406,1012300,1028246,998967,12853203,32.74,277915,251112,270755,296096,292381,298196,300005,301903,301896,297710,295627,292338,287592,357200,8.29
2,3-MONTHLY,1222542,1078884,1158549,1255732,1256808,1320242,1371764,1508074,1452403,1347448,1335211,1351868,1396407,17055932,43.44,629568,566426,610547,655260,664732,682257,701208,763919,731352,677641,673639,665933,680463,2149951,49.92
3,4-OTHER,268015,234211,219685,180279,168880,191867,196781,195535,298530,367141,407326,468705,529159,3726114,9.49,156185,139532,135952,118278,112685,123395,126465,129718,185755,207502,227153,248263,274815,1612817,37.45
4,5-INACTIVE,60001,54443,58960,58965,57077,60997,63744,67498,67772,67238,67346,69411,72061,825513,2.10,29300,27499,29900,28043,27708,28598,30092,31685,32282,31443,31948,32647,34158,141914,3.29


In [30]:
cx_data0.to_clipboard(index=False)

### Office bin

In [3]:
cx_data15 = f'''
    with base as (
    
        select
            customer_id,
            segment,
            -- total_net_orders,
            -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin

        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin

        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        office_bin,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data15 = pd.read_sql(cx_data15, conn3)
cx_data15.head()

,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,a. no-office-transit,666104,591189,622491,637500,633557,657941,676143,731607,747780,731574,738853,772263,838067,9045069,23.04,367068,336110,353209,353089,358539,364388,373693,405221,415482,401003,409692,415883,447436,2084168,48.39
1,b. office ride,1758228,1478582,1610594,1827139,1749136,1897975,1959193,2025764,2044328,2007511,1996628,2051352,2038312,24444742,62.26,576899,507560,544568,595106,586028,614893,626943,653005,663399,647608,650260,656147,657753,1496687,34.75
2,c. transit ride,418068,386909,412426,422309,420330,434409,444844,470809,476018,464721,464198,472664,484270,5771975,14.70,192050,181770,192269,193909,197215,197678,201759,213578,216926,210105,212689,211107,215133,726204,16.86


In [4]:
cx_data15

,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,a. no-office-transit,666104,591189,622491,637500,633557,657941,676143,731607,747780,731574,738853,772263,838067,9045069,23.04,367068,336110,353209,353089,358539,364388,373693,405221,415482,401003,409692,415883,447436,2084168,48.39
1,b. office ride,1758228,1478582,1610594,1827139,1749136,1897975,1959193,2025764,2044328,2007511,1996628,2051352,2038312,24444742,62.26,576899,507560,544568,595106,586028,614893,626943,653005,663399,647608,650260,656147,657753,1496687,34.75
2,c. transit ride,418068,386909,412426,422309,420330,434409,444844,470809,476018,464721,464198,472664,484270,5771975,14.70,192050,181770,192269,193909,197215,197678,201759,213578,216926,210105,212689,211107,215133,726204,16.86


In [7]:
cx_data15.to_clipboard(index=False)

In [5]:
cx_data16 = f'''
    with base as (
    
        select
            customer_id,
            segment,
            -- total_net_orders,
            -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        segment,
        office_bin,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1,2
    order by 1,2
       
'''
cx_data16 = pd.read_sql(cx_data16, conn3)
cx_data16.head()

,segment,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,a. no-office-transit,56001,44196,50935,61968,57859,65799,67169,66193,64992,64546,61062,63370,60158,784248,2.00,9566,8631,9451,10235,10323,10589,10593,10553,10502,10417,10297,10198,9999,11017,0.26
1,1-DAILY,b. office ride,530036,439153,485220,577133,548643,595295,614452,610952,603900,598079,583201,586357,559831,7332252,18.68,78690,72238,76542,82691,82630,84038,84381,84308,84114,83612,83046,82529,80764,87315,2.03
2,1-DAILY,c. transit ride,57516,48813,55335,63853,59944,67044,67800,68113,67147,66930,63224,64900,62586,813205,2.07,9392,8756,9381,9923,9970,10149,10165,10183,10124,10052,9963,9846,9685,10520,0.24
3,2-WEEKLY,a. no-office-transit,87620,75788,83135,92326,88837,93595,94070,96486,94959,93507,90119,92044,91723,1174209,2.99,33423,30685,33110,35643,35483,35813,36007,36427,36262,35861,35314,34604,34297,45730,1.06
4,2-WEEKLY,b. office ride,485476,411853,456178,517320,491144,523248,533188,542080,543640,527052,520774,527796,516802,6596551,16.80,159922,143075,154332,170239,167221,171217,172408,173622,173932,171169,170272,168756,166144,208987,4.85


In [6]:
cx_data16

,segment,office_bin,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,a. no-office-transit,56001,44196,50935,61968,57859,65799,67169,66193,64992,64546,61062,63370,60158,784248,2.00,9566,8631,9451,10235,10323,10589,10593,10553,10502,10417,10297,10198,9999,11017,0.26
1,1-DAILY,b. office ride,530036,439153,485220,577133,548643,595295,614452,610952,603900,598079,583201,586357,559831,7332252,18.68,78690,72238,76542,82691,82630,84038,84381,84308,84114,83612,83046,82529,80764,87315,2.03
2,1-DAILY,c. transit ride,57516,48813,55335,63853,59944,67044,67800,68113,67147,66930,63224,64900,62586,813205,2.07,9392,8756,9381,9923,9970,10149,10165,10183,10124,10052,9963,9846,9685,10520,0.24
3,2-WEEKLY,a. no-office-transit,87620,75788,83135,92326,88837,93595,94070,96486,94959,93507,90119,92044,91723,1174209,2.99,33423,30685,33110,35643,35483,35813,36007,36427,36262,35861,35314,34604,34297,45730,1.06
4,2-WEEKLY,b. office ride,485476,411853,456178,517320,491144,523248,533188,542080,543640,527052,520774,527796,516802,6596551,16.80,159922,143075,154332,170239,167221,171217,172408,173622,173932,171169,170272,168756,166144,208987,4.85
5,2-WEEKLY,c. transit ride,84504,77685,86181,90179,87507,91111,92368,94287,94462,90862,89935,90016,89140,1158237,2.95,31103,29569,31826,33030,32928,33248,33557,33817,33759,33104,33050,32383,31946,41570,0.97
6,3-MONTHLY,a. no-office-transit,358892,326781,348897,367007,373754,375443,387581,437591,411823,374560,369695,370305,394554,4896883,12.47,215079,198538,213272,223565,230500,230048,236177,263269,245674,223282,221649,216558,226325,893593,20.75
7,3-MONTHLY,b. office ride,633594,534874,578436,651091,636055,692535,722139,784698,767180,718988,712779,730022,744356,8906747,22.69,292625,252349,273518,304654,302042,320766,330318,354933,346264,324817,322659,323234,326260,849495,19.72
8,3-MONTHLY,c. transit ride,220745,208883,222549,226827,233323,233391,240888,264747,253721,234903,234218,233353,240279,3047827,7.76,120732,114568,122762,125803,130292,129098,132232,143289,137139,127457,127290,124119,125929,404101,9.38
9,4-OTHER,a. no-office-transit,145410,127083,121068,99874,97128,106393,109414,112151,156317,179435,198261,225662,268703,1946899,4.96,97189,86912,84952,73348,72048,77472,79549,83013,110735,119467,130161,141738,162819,1055615,24.51


In [8]:
cx_data16.to_clipboard(index=False)

In [6]:
cx_data01 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2 
'''
cx_data11 = pd.read_sql(cx_data01, conn1)
cx_data11.head()

,segment,RHA_Check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,NRHA,43435,40966,44425,47932,46199,48847,49876,50168,50305,49495,47858,47966,46081,613553,1.56,5321,5309,5474,5551,5549,5581,5586,5575,5576,5556,5518,5449,5358,5648,0.13
1,1-DAILY,RHA,213359,183969,203896,228988,215861,233257,238702,236545,233255,232616,226192,226738,218887,2892265,7.37,26216,25011,26153,26975,26853,27025,27057,27051,27007,26955,26857,26669,26252,27404,0.64
2,1-DAILY,UNKNOWN,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30,11512,10551,11265,11901,11874,11907,11982,11953,11939,11909,11899,11838,11684,12125,0.28
3,2-WEEKLY,NRHA,130978,123475,135182,144876,138939,144954,147457,150604,150714,145884,141398,142685,139058,1836204,4.68,40604,39846,42208,43916,43583,43823,44195,44576,44420,43860,43360,42631,41735,52335,1.22
4,2-WEEKLY,RHA,563960,471863,525460,603349,571767,616316,626630,635698,632173,616935,605361,613577,597579,7680668,19.56,167733,151622,163885,178100,175907,179603,180538,181811,181694,179096,178180,176034,173237,215469,5.00


In [7]:
cx_data11

,segment,RHA_Check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,NRHA,43435,40966,44425,47932,46199,48847,49876,50168,50305,49495,47858,47966,46081,613553,1.56,5321,5309,5474,5551,5549,5581,5586,5575,5576,5556,5518,5449,5358,5648,0.13
1,1-DAILY,RHA,213359,183969,203896,228988,215861,233257,238702,236545,233255,232616,226192,226738,218887,2892265,7.37,26216,25011,26153,26975,26853,27025,27057,27051,27007,26955,26857,26669,26252,27404,0.64
2,1-DAILY,UNKNOWN,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30,11512,10551,11265,11901,11874,11907,11982,11953,11939,11909,11899,11838,11684,12125,0.28
3,2-WEEKLY,NRHA,130978,123475,135182,144876,138939,144954,147457,150604,150714,145884,141398,142685,139058,1836204,4.68,40604,39846,42208,43916,43583,43823,44195,44576,44420,43860,43360,42631,41735,52335,1.22
4,2-WEEKLY,RHA,563960,471863,525460,603349,571767,616316,626630,635698,632173,616935,605361,613577,597579,7680668,19.56,167733,151622,163885,178100,175907,179603,180538,181811,181694,179096,178180,176034,173237,215469,5.00
5,2-WEEKLY,UNKNOWN,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50,69578,59644,64662,74080,72891,74770,75272,75516,75782,74754,74087,73673,72620,89396,2.08
6,3-MONTHLY,NRHA,278174,269134,290240,306550,311718,305334,314792,352564,335502,304852,302123,301452,317520,3989955,10.16,146012,142259,153705,162421,167619,163001,166174,184275,174263,159064,158207,154347,160048,539805,12.53
7,3-MONTHLY,RHA,702580,616352,657292,706374,707780,752412,780817,854893,825173,768211,765705,774469,796712,9708770,24.73,364069,325577,349216,372130,377322,390972,401786,435329,417524,388300,387837,383322,389857,1222774,28.39
8,3-MONTHLY,UNKNOWN,241788,193398,211017,242808,237310,262496,276155,300617,291728,274385,267383,275947,282175,3357207,8.55,119487,98590,107626,120709,119791,128284,133248,144315,139565,130277,127595,128264,130558,387372,8.99
9,4-OTHER,NRHA,85742,78361,74460,63071,60109,64727,66126,66296,97265,116703,128350,146335,166314,1213859,3.09,51505,48060,47309,42446,40978,42578,43607,44915,62447,68900,75468,82297,91540,563947,13.09


In [12]:
cx_data11.to_clipboard(index=False)

In [10]:
cx_data02 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data12 = pd.read_sql(cx_data02, conn1)
cx_data12.head()

,segment,Zwigato_check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,Non-Zwigato,54715,51736,55616,60228,57394,61772,63119,63102,62506,62040,59676,59232,56850,767986,1.96,6747,6697,6919,7041,7040,7092,7083,7084,7054,7050,6998,6890,6778,7181,0.17
1,1-DAILY,UNKNOWN,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30,11512,10551,11265,11901,11874,11907,11982,11953,11939,11909,11899,11838,11684,12125,0.28
2,1-DAILY,Zwigato,202079,173199,192705,216692,204666,220332,225459,223611,221054,220071,214374,215472,208118,2737832,6.97,24790,23623,24708,25485,25362,25514,25560,25542,25529,25461,25377,25228,24832,25871,0.60
3,2-WEEKLY,Non-Zwigato,157587,146771,159508,172755,165407,174601,177703,180962,180375,175053,168762,169455,164385,2193324,5.59,49120,47913,50911,53030,52832,53246,53535,54018,53753,53127,52231,51303,50244,63992,1.49
4,2-WEEKLY,UNKNOWN,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50,69578,59644,64662,74080,72891,74770,75272,75516,75782,74754,74087,73673,72620,89396,2.08


In [11]:
cx_data12

,segment,Zwigato_check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,Non-Zwigato,54715,51736,55616,60228,57394,61772,63119,63102,62506,62040,59676,59232,56850,767986,1.96,6747,6697,6919,7041,7040,7092,7083,7084,7054,7050,6998,6890,6778,7181,0.17
1,1-DAILY,UNKNOWN,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30,11512,10551,11265,11901,11874,11907,11982,11953,11939,11909,11899,11838,11684,12125,0.28
2,1-DAILY,Zwigato,202079,173199,192705,216692,204666,220332,225459,223611,221054,220071,214374,215472,208118,2737832,6.97,24790,23623,24708,25485,25362,25514,25560,25542,25529,25461,25377,25228,24832,25871,0.60
3,2-WEEKLY,Non-Zwigato,157587,146771,159508,172755,165407,174601,177703,180962,180375,175053,168762,169455,164385,2193324,5.59,49120,47913,50911,53030,52832,53246,53535,54018,53753,53127,52231,51303,50244,63992,1.49
4,2-WEEKLY,UNKNOWN,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50,69578,59644,64662,74080,72891,74770,75272,75516,75782,74754,74087,73673,72620,89396,2.08
5,2-WEEKLY,Zwigato,537351,448567,501134,575470,545299,586669,596384,605340,602512,587766,577997,586807,572252,7323548,18.65,159217,143555,155182,168986,166658,170180,171198,172369,172361,169829,169309,167362,164728,203812,4.73
6,3-MONTHLY,Non-Zwigato,327950,316249,334932,356064,364065,362188,372820,414246,392940,356443,348802,347532,363362,4657593,11.86,173509,169091,180137,190426,198039,194561,198735,218241,206026,187621,184421,179623,184769,646564,15.01
7,3-MONTHLY,UNKNOWN,241788,193398,211017,242808,237310,262496,276155,300617,291728,274385,267383,275947,282175,3357207,8.55,119487,98590,107626,120709,119791,128284,133248,144315,139565,130277,127595,128264,130558,387372,8.99
8,3-MONTHLY,Zwigato,652804,569237,612600,656860,655433,695558,722789,793211,767735,716620,719026,728389,750870,9041132,23.03,336572,298745,322784,344125,346902,359412,369225,401363,385761,359743,361623,358046,365136,1116015,25.91
9,4-OTHER,Non-Zwigato,104599,95229,87306,73338,70737,77427,78561,77839,113819,136872,150202,169375,193174,1428478,3.64,63520,59143,56489,50244,49015,51973,52764,53984,74517,82258,89525,96588,107818,677463,15.73


In [13]:
cx_data12.to_clipboard(index=False)

In [14]:
cx_data03 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data13 = pd.read_sql(cx_data03, conn1)
cx_data13.head()

,segment,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,1. BOTH,180753,153534,171183,193602,182388,196992,201621,199540,197117,196249,191360,192096,185769,2442204,6.22,22201,21061,22063,22809,22690,22832,22872,22863,22843,22784,22715,22584,22231,23153,0.54
1,1-DAILY,2. ONLY Zwigato,21326,19665,21522,23090,22278,23340,23838,24071,23937,23822,23014,23376,22349,295628,0.75,2589,2562,2645,2676,2672,2682,2688,2679,2686,2677,2662,2644,2601,2718,0.06
2,1-DAILY,3. ONLY RHA,32606,30435,32713,35386,33473,36265,37081,37005,36138,36367,34832,34642,33118,450061,1.15,4015,3950,4090,4166,4163,4193,4185,4188,4164,4171,4142,4085,4021,4251,0.10
3,1-DAILY,4. NONE,22109,21301,22903,24842,23921,25507,26038,26097,26368,25673,24844,24590,23732,317925,0.81,2732,2747,2829,2875,2877,2899,2898,2896,2890,2879,2856,2805,2757,2930,0.07
4,1-DAILY,5. IOS,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30,11512,10551,11265,11901,11874,11907,11982,11953,11939,11909,11899,11838,11684,12125,0.28


In [15]:
cx_data13

,segment,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1-DAILY,1. BOTH,180753,153534,171183,193602,182388,196992,201621,199540,197117,196249,191360,192096,185769,2442204,6.22,22201,21061,22063,22809,22690,22832,22872,22863,22843,22784,22715,22584,22231,23153,0.54
1,1-DAILY,2. ONLY Zwigato,21326,19665,21522,23090,22278,23340,23838,24071,23937,23822,23014,23376,22349,295628,0.75,2589,2562,2645,2676,2672,2682,2688,2679,2686,2677,2662,2644,2601,2718,0.06
2,1-DAILY,3. ONLY RHA,32606,30435,32713,35386,33473,36265,37081,37005,36138,36367,34832,34642,33118,450061,1.15,4015,3950,4090,4166,4163,4193,4185,4188,4164,4171,4142,4085,4021,4251,0.10
3,1-DAILY,4. NONE,22109,21301,22903,24842,23921,25507,26038,26097,26368,25673,24844,24590,23732,317925,0.81,2732,2747,2829,2875,2877,2899,2898,2896,2890,2879,2856,2805,2757,2930,0.07
4,1-DAILY,5. IOS,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30,11512,10551,11265,11901,11874,11907,11982,11953,11939,11909,11899,11838,11684,12125,0.28
5,2-WEEKLY,1. BOTH,476115,391856,438500,508322,481059,519297,527951,535027,532528,519905,511999,519939,506188,6468686,16.48,140449,125395,135913,148863,146787,150047,150917,151932,151956,149747,149319,147607,145280,179836,4.18
6,2-WEEKLY,2. ONLY Zwigato,61236,56711,62634,67148,64240,67372,68433,70313,69984,67861,65998,66868,66064,854862,2.18,18768,18160,19269,20123,19871,20133,20281,20437,20405,20082,19990,19755,19448,23976,0.56
7,2-WEEKLY,3. ONLY RHA,87845,80007,86960,95027,90708,97019,98679,100671,99645,97030,93362,93638,91391,1211982,3.09,27284,26227,27972,29237,29120,29556,29621,29879,29738,29349,28861,28427,27957,35633,0.83
8,2-WEEKLY,4. NONE,69742,66764,72548,77728,74699,77582,79024,80291,80730,78023,75400,75817,72994,981342,2.50,21836,21686,22939,23793,23712,23690,23914,24139,24015,23778,23370,22876,22287,28359,0.66
9,2-WEEKLY,5. IOS,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50,69578,59644,64662,74080,72891,74770,75272,75516,75782,74754,74087,73673,72620,89396,2.08


In [20]:
cx_data13.to_clipboard(index=False)

In [17]:
cx_data04 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        -- segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1
'''
cx_data14 = pd.read_sql(cx_data04, conn1)
cx_data14.head()

,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1. BOTH,1332944,1124914,1220438,1339247,1293611,1398846,1438524,1500658,1515797,1485249,1489885,1532691,1551282,18224086,46.42,514080,453813,485039,509743,507036,529102,540146,568324,577726,561062,569498,573506,585186,1729074,40.15
1,2. ONLY Zwigato,221330,206384,223950,229766,223963,230130,236731,254799,257812,252364,254537,262706,272237,3126709,7.96,95364,90804,97159,98867,98318,98606,100874,108650,110401,107339,110145,111199,115297,399112,9.27
2,3. ONLY RHA,331119,307758,320283,332713,327925,345702,354137,375008,377424,369480,365861,373703,385168,4566281,11.63,146707,139996,144812,146890,149409,152294,155221,164557,167020,161933,162607,162779,168281,641938,14.90
3,4. NONE,330352,318431,334150,346235,346003,347094,355799,379941,391301,379738,380267,391575,413637,4714523,12.01,155279,151838,159168,162598,166261,163256,166239,178647,184408,177897,180481,181857,192212,805402,18.70
4,5. IOS,626655,499193,546690,638987,611521,668553,694989,717774,725792,716975,709129,735604,738325,8630187,21.98,224587,188989,203868,224006,220758,233701,239915,251626,256252,250485,249910,253796,259346,731533,16.98


In [18]:
cx_data14

,tech_savvy,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,1. BOTH,1332944,1124914,1220438,1339247,1293611,1398846,1438524,1500658,1515797,1485249,1489885,1532691,1551282,18224086,46.42,514080,453813,485039,509743,507036,529102,540146,568324,577726,561062,569498,573506,585186,1729074,40.15
1,2. ONLY Zwigato,221330,206384,223950,229766,223963,230130,236731,254799,257812,252364,254537,262706,272237,3126709,7.96,95364,90804,97159,98867,98318,98606,100874,108650,110401,107339,110145,111199,115297,399112,9.27
2,3. ONLY RHA,331119,307758,320283,332713,327925,345702,354137,375008,377424,369480,365861,373703,385168,4566281,11.63,146707,139996,144812,146890,149409,152294,155221,164557,167020,161933,162607,162779,168281,641938,14.90
3,4. NONE,330352,318431,334150,346235,346003,347094,355799,379941,391301,379738,380267,391575,413637,4714523,12.01,155279,151838,159168,162598,166261,163256,166239,178647,184408,177897,180481,181857,192212,805402,18.70
4,5. IOS,626655,499193,546690,638987,611521,668553,694989,717774,725792,716975,709129,735604,738325,8630187,21.98,224587,188989,203868,224006,220758,233701,239915,251626,256252,250485,249910,253796,259346,731533,16.98


In [19]:
cx_data14.to_clipboard(index=False)

### regularity_segment

In [5]:
cx_data = f'''
    with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status='dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_regularity_segment regularity_segment,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data1 = pd.read_sql(cx_data, conn2)
cx_data1.head()

KeyboardInterrupt: 

In [ ]:
cx_data1

In [15]:
cx_data1.to_clipboard()

### ps_tag_link

In [15]:
cx_data2 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_link
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Link','Bike Lite','Bike Pink','Bike Metro')
        -- and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        ps_tag_link ps_tag_link,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data2 = pd.read_sql(cx_data2, conn4)
cx_data2.head()

,ps_tag_link,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NPS,200950,186185,207788,240930,216769,232863,238329,246350,243932,237578,235503,230429,230196,2947802,25.68,72130,68492,76087,86402,81136,84249,84684,89298,88919,84563,84674,82707,82665,206714,8.89
1,PS,95493,85435,94431,106785,98839,108295,111892,114053,115425,113973,114787,112284,112276,1383968,12.06,40072,37239,41118,45765,43840,46516,47428,49272,49966,48166,48360,47064,46792,135039,5.81
2,UNKNOWN,519308,436601,462724,497516,474595,533636,559889,582714,590582,609548,618657,618483,643125,7147378,62.26,311533,269614,286267,303494,293855,319093,331072,349136,355936,359002,366229,364224,380021,1982851,85.30


In [16]:
cx_data2

,ps_tag_link,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NPS,200950,186185,207788,240930,216769,232863,238329,246350,243932,237578,235503,230429,230196,2947802,25.68,72130,68492,76087,86402,81136,84249,84684,89298,88919,84563,84674,82707,82665,206714,8.89
1,PS,95493,85435,94431,106785,98839,108295,111892,114053,115425,113973,114787,112284,112276,1383968,12.06,40072,37239,41118,45765,43840,46516,47428,49272,49966,48166,48360,47064,46792,135039,5.81
2,UNKNOWN,519308,436601,462724,497516,474595,533636,559889,582714,590582,609548,618657,618483,643125,7147378,62.26,311533,269614,286267,303494,293855,319093,331072,349136,355936,359002,366229,364224,380021,1982851,85.30


In [18]:
cx_data2.to_clipboard(index=False)

### ps_tag_auto

In [20]:
cx_data3 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_auto
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium')
        -- and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        ps_tag_auto ps_tag_auto,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data3 = pd.read_sql(cx_data3, conn4)
cx_data3.head()

,ps_tag_auto,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,PS,488470,451744,477476,491756,480636,458719,436384,447086,442864,424876,418011,432110,429845,5879977,24.82,199417,196086,204184,201960,204672,185164,172659,178083,177337,167422,167479,168387,168956,524378,16.44
1,UNKNOWN,367589,278594,302230,345652,351235,440532,518059,594273,629167,626178,637780,690304,728536,6510129,27.48,234261,191641,205151,218168,226795,264070,297145,337330,354428,344341,353740,370277,391740,1869649,58.62
2,NPS,889683,737662,799209,898554,863873,896738,889148,910337,904906,874272,862552,890853,879323,11297110,47.69,320315,285216,303920,324405,323034,319810,313153,322676,321827,307396,307602,307905,308550,795368,24.94


In [21]:
cx_data3

,ps_tag_auto,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,PS,488470,451744,477476,491756,480636,458719,436384,447086,442864,424876,418011,432110,429845,5879977,24.82,199417,196086,204184,201960,204672,185164,172659,178083,177337,167422,167479,168387,168956,524378,16.44
1,UNKNOWN,367589,278594,302230,345652,351235,440532,518059,594273,629167,626178,637780,690304,728536,6510129,27.48,234261,191641,205151,218168,226795,264070,297145,337330,354428,344341,353740,370277,391740,1869649,58.62
2,NPS,889683,737662,799209,898554,863873,896738,889148,910337,904906,874272,862552,890853,879323,11297110,47.69,320315,285216,303920,324405,323034,319810,313153,322676,321827,307396,307602,307905,308550,795368,24.94


In [13]:
cx_data3.to_clipboard(index=False)

### taxi_income_segment

In [41]:
cx_data4 = f'''
    with base as (
    select 
        customer_id,
        taxi_income_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_income_segment taxi_income_segment,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data4 = pd.read_sql(cx_data4, conn1)
cx_data4.head()

,taxi_income_segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,HIGH_INCOME,1425044,1211912,1310382,1445645,1393608,1502188,1548466,1612807,1628498,1599188,1592357,1639872,1658363,19568330,49.84,557612,497647,530658,560561,555820,579207,591795,622955,632701,615196,620634,625005,637659,1997083,46.37
1,MEDIUM_INCOME,643410,584722,619908,653558,640742,669396,683737,724357,732508,711756,713292,732662,752666,8862714,22.57,272469,257328,269795,276667,279463,282116,286669,304688,310306,299211,303494,304441,313926,1136600,26.39
2,UNKNOWN,662949,555967,606122,673657,655514,704905,732120,766911,779460,770500,772473,800198,819041,9299817,23.69,258347,224441,241612,255992,256541,266641,274315,291023,297945,292094,295936,301439,313239,957009,22.22
3,LOW_INCOME,111003,104082,109099,114089,113162,113836,115859,124106,127662,122362,121560,123552,130582,1530954,3.90,47595,46027,47981,48885,49961,48995,49618,53139,54857,52215,52580,52257,55501,216396,5.02


In [57]:
cx_data4

,taxi_income_segment,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,HIGH_INCOME,1425044,1211912,1310382,1445645,1393608,1502188,1548466,1612807,1628498,1599188,1592357,1639872,1658363,19568330,49.84,557612,497647,530658,560561,555820,579207,591795,622955,632701,615196,620634,625005,637659,1997083,46.37
1,MEDIUM_INCOME,643410,584722,619908,653558,640742,669396,683737,724357,732508,711756,713292,732662,752666,8862714,22.57,272469,257328,269795,276667,279463,282116,286669,304688,310306,299211,303494,304441,313926,1136600,26.39
2,UNKNOWN,662949,555967,606122,673657,655514,704905,732120,766911,779460,770500,772473,800198,819041,9299817,23.69,258347,224441,241612,255992,256541,266641,274315,291023,297945,292094,295936,301439,313239,957009,22.22
3,LOW_INCOME,111003,104082,109099,114089,113162,113836,115859,124106,127662,122362,121560,123552,130582,1530954,3.90,47595,46027,47981,48885,49961,48995,49618,53139,54857,52215,52580,52257,55501,216396,5.02


In [17]:
cx_data4.to_clipboard(index=False)

### taxi_service_affinity

In [42]:
cx_data5 = f'''
    with base as (
    select 
        customer_id,
        taxi_service_affinity
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_service_affinity taxi_service_affinity,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data5 = pd.read_sql(cx_data5, conn1)
cx_data5.head()

,taxi_service_affinity,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NO_AFFINITY,271561,254934,278523,293430,295140,308562,321023,333814,336815,328107,329027,330103,331313,4012352,10.22,83900,79363,85444,89766,90830,93410,95552,98459,99609,97074,97112,95943,95525,144935,3.37
1,AUTO_CAB,63088,56466,59397,65981,65734,70139,71919,74234,75495,73705,71965,74294,74044,896461,2.28,22689,20700,21968,23640,24094,24881,25434,26192,26817,25957,25843,25860,25650,46468,1.08
2,LINK_CAB,3975,3927,4276,4530,4610,4686,4766,5280,5519,5499,5348,5462,5680,63558,0.16,2170,2107,2271,2412,2460,2516,2586,2737,2863,2787,2764,2730,2751,6231,0.14
3,ONLY_CAB,76698,69221,74654,88489,86182,94549,95011,96028,97243,93824,90814,93927,93525,1150165,2.93,27525,25644,27742,30669,30979,32599,32794,33669,34405,33303,32905,32755,32511,65631,1.52
4,ONLY_AUTO,1191479,984155,1074816,1240334,1183967,1290145,1324068,1365931,1374603,1355696,1333480,1381611,1361871,16462156,41.93,349352,311827,334674,366354,365978,380960,386696,399961,403183,394195,393177,392724,390791,640049,14.86


In [61]:
cx_data5

,taxi_service_affinity,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NO_AFFINITY,271561,254934,278523,293430,295140,308562,321023,333814,336815,328107,329027,330103,331313,4012352,10.22,83900,79363,85444,89766,90830,93410,95552,98459,99609,97074,97112,95943,95525,144935,3.37
1,AUTO_CAB,63088,56466,59397,65981,65734,70139,71919,74234,75495,73705,71965,74294,74044,896461,2.28,22689,20700,21968,23640,24094,24881,25434,26192,26817,25957,25843,25860,25650,46468,1.08
2,LINK_CAB,3975,3927,4276,4530,4610,4686,4766,5280,5519,5499,5348,5462,5680,63558,0.16,2170,2107,2271,2412,2460,2516,2586,2737,2863,2787,2764,2730,2751,6231,0.14
3,ONLY_CAB,76698,69221,74654,88489,86182,94549,95011,96028,97243,93824,90814,93927,93525,1150165,2.93,27525,25644,27742,30669,30979,32599,32794,33669,34405,33303,32905,32755,32511,65631,1.52
4,ONLY_AUTO,1191479,984155,1074816,1240334,1183967,1290145,1324068,1365931,1374603,1355696,1333480,1381611,1361871,16462156,41.93,349352,311827,334674,366354,365978,380960,386696,399961,403183,394195,393177,392724,390791,640049,14.86
5,UNKNOWN,695658,600209,620590,607277,605824,616808,637298,702726,724616,706126,725485,758672,846530,8847819,22.54,471914,418717,436772,433700,433134,440076,452621,496260,512308,494529,508878,522576,564709,3033495,70.43
6,ONLY_LINK,425676,384096,419305,465936,443220,479210,496188,513029,514475,505866,507265,512413,506344,6173023,15.72,135555,126975,137824,149601,148166,154663,158014,163459,164961,160808,161626,160443,158231,282104,6.55
7,LINK_AUTO,114271,103675,113950,120972,118349,126226,129909,137139,139362,134983,136298,139802,141345,1656281,4.22,42918,40110,43351,45963,46144,47854,48700,51068,51663,50063,50339,50111,50157,88175,2.05


### RHA Check

In [43]:
cx_data6 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data6 = pd.read_sql(cx_data6, conn1)
cx_data6.head()

,RHA_Check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,NRHA,548623,522559,555232,573336,568018,574641,589878,631868,647282,630896,634523,654946,691053,7822855,19.92,249313,241499,255155,260351,263665,260874,266203,286179,294155,284751,290407,293174,309117,1203396,27.94
1,RHA,1665048,1433183,1541684,1672499,1621501,1745114,1793116,1876233,1892782,1853671,1853702,1903355,1928712,22780600,58.02,661318,594251,630263,656956,656603,681568,695454,733103,744520,722609,731412,735230,750865,2369229,55.01
2,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05


### Zwigato_check

In [44]:
cx_data7 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
'''
cx_data7 = pd.read_sql(cx_data7, conn1)
cx_data7.head()

,Zwigato_check,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,Zwigato,1552516,1329657,1442052,1566445,1514891,1626588,1672537,1752345,1769714,1733222,1738978,1789485,1813418,21301848,54.26,608349,543764,581229,607402,603987,626487,639773,675489,686151,666253,677017,682021,696431,2119252,49.20
1,Non-Zwigato,661155,626085,654864,679390,674628,693167,710457,755756,770350,751345,749247,768816,806347,9301607,23.69,302282,291986,304189,309905,316281,315955,321884,343793,352524,341107,344802,346383,363551,1453373,33.74
2,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05


### Office Usecases

In [45]:
cx_data8 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams|intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag    
        from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(office_tag, 'UNKNOWN') office_tag,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,

        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data8 = pd.read_sql(cx_data8, conn1)
cx_data8.head()

,office_tag,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05
1,no-office-apps,1469808,1355749,1432430,1492242,1479678,1528872,1558964,1649768,1679274,1638774,1634308,1671615,1735623,20327105,51.77,616557,585519,613803,623132,636123,637323,647204,687838,704220,680637,688335,689067,717155,2556038,59.34
2,office-apps,743863,599993,664486,753593,709841,790883,824030,858333,860790,845793,853917,886686,884142,10276350,26.17,294074,250231,271615,294175,284145,305119,314453,331444,334455,326723,333484,339337,342827,1016587,23.60


In [63]:
cx_data8

,office_tag,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,UNKNOWN,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05,225392,189693,204628,224798,221517,234517,240740,252523,257134,251356,250825,254738,260343,734463,17.05
1,no-office-apps,1469808,1355749,1432430,1492242,1479678,1528872,1558964,1649768,1679274,1638774,1634308,1671615,1735623,20327105,51.77,616557,585519,613803,623132,636123,637323,647204,687838,704220,680637,688335,689067,717155,2556038,59.34
2,office-apps,743863,599993,664486,753593,709841,790883,824030,858333,860790,845793,853917,886686,884142,10276350,26.17,294074,250231,271615,294175,284145,305119,314453,331444,334455,326723,333484,339337,342827,1016587,23.60


### Age Group

In [46]:
cx_data9 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    age as (
    
    select
        customer_id,
        age_group
    from 
        hive.experiments_internal.customer_predicated_age_immutable
    )
    
    
    select
        coalesce(age_group, 'UNKNOWN') age_group,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,
        
        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,
        
        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct

      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        age
        on base.customer_id = age.customer_id
    group by 1
    order by 1
'''
cx_data9 = pd.read_sql(cx_data9, conn1)
cx_data9.head(10)

,age_group,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,21-25,509873,456853,499531,522643,521816,534225,548593,577548,568701,543323,547984,552993,573680,6957763,17.72,221700,204949,222803,225712,231473,230347,234649,246403,240743,226812,231866,228207,236646,856025,19.87
1,26-30,478597,428342,465846,491231,472351,499278,517382,543320,535370,513466,520225,529711,535016,6530135,16.63,197954,181369,195244,201546,197729,203538,208682,218557,214853,205502,208597,209633,213100,695672,16.15
2,31-35,406287,359524,379016,411077,397284,425582,436771,452920,444428,428970,424217,434698,437711,5438485,13.85,164710,150446,156961,165656,163672,170124,173073,179911,175374,168453,167928,169014,171452,565804,13.14
3,36-45,601464,522688,550682,601850,583558,630934,645042,666384,649062,631614,615268,628920,629256,7956722,20.27,239093,219579,227105,237909,239250,247494,252018,261069,251325,243161,240417,239086,241598,830405,19.28
4,Above-45,129640,111252,118444,129918,127440,137886,141381,147803,142852,139833,135707,138500,140540,1741196,4.43,51701,46953,48516,50929,51920,53971,55519,57923,55508,53495,53118,52805,53749,189354,4.40
5,Below-21,39670,35839,38924,40314,39501,40691,41541,43975,41438,39661,39714,40278,41576,523122,1.33,16738,15815,16991,16901,17531,17571,17763,18581,17480,16483,16819,16504,17096,63587,1.48
6,UNKNOWN,676875,542185,593068,689916,661076,721729,749472,796231,886277,906939,916567,971184,1002873,10114392,25.76,244127,206332,222426,243452,240210,253914,260693,289361,340526,344810,353899,367893,386684,1106241,25.68


In [65]:
cx_data9

,age_group,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,21-25,509873,456853,499531,522643,521816,534225,548593,577548,568701,543323,547984,552993,573680,6957763,17.72,221700,204949,222803,225712,231473,230347,234649,246403,240743,226812,231866,228207,236646,856025,19.87
1,26-30,478597,428342,465846,491231,472351,499278,517382,543320,535370,513466,520225,529711,535016,6530135,16.63,197954,181369,195244,201546,197729,203538,208682,218557,214853,205502,208597,209633,213100,695672,16.15
2,31-35,406287,359524,379016,411077,397284,425582,436771,452920,444428,428970,424217,434698,437711,5438485,13.85,164710,150446,156961,165656,163672,170124,173073,179911,175374,168453,167928,169014,171452,565804,13.14
3,36-45,601464,522688,550682,601850,583558,630934,645042,666384,649062,631614,615268,628920,629256,7956722,20.27,239093,219579,227105,237909,239250,247494,252018,261069,251325,243161,240417,239086,241598,830405,19.28
4,Above-45,129640,111252,118444,129918,127440,137886,141381,147803,142852,139833,135707,138500,140540,1741196,4.43,51701,46953,48516,50929,51920,53971,55519,57923,55508,53495,53118,52805,53749,189354,4.40
5,Below-21,39670,35839,38924,40314,39501,40691,41541,43975,41438,39661,39714,40278,41576,523122,1.33,16738,15815,16991,16901,17531,17571,17763,18581,17480,16483,16819,16504,17096,63587,1.48
6,UNKNOWN,676875,542185,593068,689916,661076,721729,749472,796231,886277,906939,916567,971184,1002873,10114392,25.76,244127,206332,222426,243452,240210,253914,260693,289361,340526,344810,353899,367893,386684,1106241,25.68


### Office Use-Cases

In [47]:
cx_data11 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    use_case as (
    
    select
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level
    where 
        current_city = 'Bangalore'
        and primary_tag = 'office'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        case 
        when pickup_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        when drop_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        else 'non-office' end office_use_case,
        count(distinct order_id) orders
    from 
        orders.order_logs_snapshot
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and order_status = 'dropped'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2,3
    )
    
    select
        coalesce(office_use_case, 'non-office') office_use_case,
        sum(case when week_start_date = '20241216' then orders end ) w01,
        sum(case when week_start_date = '20241223' then orders end ) w02,
        sum(case when week_start_date = '20241230' then orders end ) w03,
        sum(case when week_start_date = '20250106' then orders end ) w04,
        sum(case when week_start_date = '20250113' then orders end ) w05,
        sum(case when week_start_date = '20250120' then orders end ) w06,
        sum(case when week_start_date = '20250127' then orders end ) w07,
        sum(case when week_start_date = '20250203' then orders end ) w08,
        sum(case when week_start_date = '20250210' then orders end ) w09,
        sum(case when week_start_date = '20250217' then orders end ) w10,
        sum(case when week_start_date = '20250224' then orders end ) w11,
        sum(case when week_start_date = '20250303' then orders end ) w12,
        sum(case when week_start_date = '20250310' then orders end ) w13,
        
        sum(orders.orders) as overall,
        100.00*sum(orders)/sum(sum(orders))over() as pct,

        count(distinct case when week_start_date = '20241216' then orders.customer_id end ) C_w01,
        count(distinct case when week_start_date = '20241223' then orders.customer_id end ) C_w02,
        count(distinct case when week_start_date = '20241230' then orders.customer_id end ) C_w03,
        count(distinct case when week_start_date = '20250106' then orders.customer_id end ) C_w04,
        count(distinct case when week_start_date = '20250113' then orders.customer_id end ) C_w05,
        count(distinct case when week_start_date = '20250120' then orders.customer_id end ) C_w06,
        count(distinct case when week_start_date = '20250127' then orders.customer_id end ) C_w07,
        count(distinct case when week_start_date = '20250203' then orders.customer_id end ) C_w08,
        count(distinct case when week_start_date = '20250210' then orders.customer_id end ) C_w09,
        count(distinct case when week_start_date = '20250217' then orders.customer_id end ) C_w10,
        count(distinct case when week_start_date = '20250224' then orders.customer_id end ) C_w11,
        count(distinct case when week_start_date = '20250303' then orders.customer_id end ) C_w12,
        count(distinct case when week_start_date = '20250310' then orders.customer_id end ) C_w13,

        count(distinct orders.customer_id) as C_overall,
        100.00*count(distinct orders.customer_id)/sum(count(distinct orders.customer_id))over() as c_pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data11 = pd.read_sql(cx_data11, conn1)
cx_data11.head(10)

,office_use_case,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,non-office,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27,1041304,953795,1016143,1045690,1051460,1077286,1102591,1166184,1186804,1149655,1165312,1170078,1206913,4126851,73.44
1,office,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73,242563,185005,196080,244612,229942,253719,257489,271786,278891,274949,272522,285146,287877,1492522,26.56


In [69]:
cx_data11

,office_use_case,w01,w02,w03,w04,w05,w06,w07,w08,w09,w10,w11,w12,w13,overall,pct,C_w01,C_w02,C_w03,C_w04,C_w05,C_w06,C_w07,C_w08,C_w09,C_w10,C_w11,C_w12,C_w13,C_overall,c_pct
0,non-office,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27,1041304,953795,1016143,1045690,1051460,1077286,1102591,1166184,1186804,1149655,1165312,1170078,1206913,4126851,73.44
1,office,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73,242563,185005,196080,244612,229942,253719,257489,271786,278891,274949,272522,285146,287877,1492522,26.56
